In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from instruments.gilson.gilson_ethernet import GilsonEthernet
from instruments.gilson.tray import Tray
from instruments.gilson.rack import Rack_209, Rack_3dp
from instruments.gilson.probe import ProbeState    # NEW
from core.logging import flow_logger as logger
from core.tracing import *
from instruments.vici.dim import DIM
from instruments.vici.fsm import FSM
from instruments.syr_pumps.HB_syr_pump import HBElite
from instruments.knauer.knauer_pump_azura import KnauerPumpAzura
from ecosystems.reaction_segment_generation import RSG
from ecosystems.segmentation import Segmentation


# Experiment framework
from core.experiment_context import ExperimentManager
from core.experiment_builder import ExperimentBuilder
from core.experiment_validator import ExperimentValidator
from core.experiment_compiler import ExperimentCompiler
from core.experiment_intent import ExperimentIntent
from core.inventory import Inventory

In [2]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4


In [3]:
# --- Instantiate the tray ---
tray = Tray()

# --- Instantiate modules for the tray ---
rack1 = Rack_209()
rack2 = Rack_3dp()

# --- Assign modules to tray slots ---
tray.assign_slot(1, rack1, alias = "rack1")    # Assigns rack1 to slot 1
tray.assign_slot(2, rack2, alias = "rack2")    # Assigns rack2 to slot 2
tray.assign_slot(3, dim, alias = "dim")      # Assigns dim to slot 3

In [4]:
# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Instantiate the probe (state machine only) ---
probe = ProbeState()

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("VB-1E-9")

In [5]:
# Instantiate the RSG ecosystem with the connected Gilson, pump, DIM and probe
rsg = RSG(gilson=g, pump=syr_pump, dim=dim, probe=probe)

# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, fsm=fsm, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [6]:
DWELL_TIME_S = 40
NUM_RUNS = 20
SPLIT_POINT = 10

In [7]:
def run_one_slug(rsg, fsm, dim, rack_index):
    # 1. prep air gap (system is already primed)
    rsg.take_air_gap(25, 0.05)
    print("[PROBE]", rsg.probe.status())

    # 2. gas prime 
    fsm.gas_to_dim()
    time.sleep(20)

    # 3. load DIM
    dim.load()

    # 4. sample pickup
    rsg.pickup_from_vial("rack1", rack_index, 100, 0.2)
    print("[PROBE]", rsg.probe.status())

    # 5. post pickup air gap
    rsg.take_air_gap(5, 0.05)
    print("[PROBE]", rsg.probe.status())

    # 6. dispense into DIM
    rsg.dispense_in_dim(0.5, 120)
    print("[PROBE]", rsg.probe.status())

    # 7. launch (CRITICAL ORDER)
    fsm.carrier_to_dim_fast()
    dim.inject_fast()

    # 8. dwell (physics only)
    time.sleep(DWELL_TIME_S)

    # 9. wash pickup
    rsg.pickup_from_vial("rack2", 1, 100, 0.4)

    # 10. wash dispense
    rsg.dispense_in_vial("rack2", 2, 110, 0.5)

    print("[PROBE FINAL]", rsg.probe.status())

In [8]:
# STARTS THE RUN

for i in range(NUM_RUNS):

    rack_index = 1 if i < SPLIT_POINT else 2

    print(f"\n========== SLUG {i+1} ==========")

    run_one_slug(rsg, fsm, dim, rack_index)


========== SLUG 1 ==========
[Air Gap] 25uL @ 0.05mL/min
[Probe] [working_fluid] [air (25.0 uL)]
[PROBE] [working_fluid] [air (25.0 uL)]
[FSM] Valve moved to A -> state = GAS_TO_DIM
[DIM] Valve moved to B -> state = DIMState.LOAD
[Pickup] 100uL from rack1 vial 1 @ 0.2mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[Probe] [working_fluid] [air (25.0 uL)] [sample (100.0 uL)]
[PROBE] [working_fluid] [air (25.0 uL)] [sample (100.0 uL)]
[Air Gap] 5uL @ 0.05mL/min
[Probe] [working_fluid] [air (25.0 uL)] [sample (100.0 uL)] [air (5.0 uL)]
[PROBE] [working_fluid] [air (25.0 uL)] [sample (100.0 uL)] [air (5.0 uL)]
[DIM Dispense] 120uL @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DIM] Physical infusion complete
[Probe] [working_fluid] [air (10.0 uL)]
[DIM] ================= COMPLETE ================

In [19]:
g.go_into_vial("rack2", 2)
syr_pump.infuse_volume(20, 0.5)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [9]:
syr_pump.infuse_volume(10, 0.5)

In [16]:
g.go_into_vial("rack2", 1)
syr_pump.withdraw_volume(100, 0.5)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [13]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [16]:
k_pump.get_sernum()

'SERNUM:FAW250900017'